<h1 align="center"><font color="red">Stateful vs Stateless Agent Design</font></h1>

<font color="pink">Senior Data Scientist.: Dr. Eddy Giusepe Chirinos Isidro</font>

# <font color="gree">Initial Setup</font>

You can find the API key in [groq](https://console.groq.com/home).

In [1]:

import os
from groq import Groq
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
 
# Get an API key in https://console.groq.com/keys and set it here:
GROQ_API_KEY = os.environ["GROQ_API_KEY"]
    
# Initializing the client:
client = Groq(api_key=GROQ_API_KEY)
 
# Using an efficient model from Groq: Llama 3.1 8B Instant
MODEL_ID = "llama-3.1-8b-instant"

An important setup decision here is the choice of a specific model. `llama-3.1-8b-instant` is highly cost-efficient model that is, at the time of writing, generously supported on `Groq's 2026 free tier`: it allows up to `14400 requests per day`. That makes it an ideal choice for illustrating the stateless and stateful agent paradigms bellow.

# <font color="gree">The Tradeoff</font>

Architectures based on `stateless agents` can be scaled horizontally with remarkable ease. Since no user memory is stored on a backend server, incoming requests can be forwarded to any available instance. There is, however, an important limitation in multi-turn conversations: the `frontend` must re-send the whole conversation history alongside every new request. As a result, the context window grows with a snowballing effect, quickly driving up token usage.

# <font color="gree">Illustrative Example</font>

This runnable code illustrates, through a basic scenario, how a stateless agent typically interacts with a `Groq` language model.

First, we define a `stateless_agent` function that emulates an agent’s interaction with our chosen model. Importantly, no state or memory of the conversation is kept internally. Instead, the previous conversation history can optionally be passed in as a parameter and appended to the current prompt. The API call to the Groq model takes place in `client.chat.completions.create()`.

In [2]:

def stateless_agent(prompt: str, provided_history: list = None) -> str:
    """
    The agent relies completely on the client to provide context.
    It retains no information from past interactions in local memory.
    """
    # Initializing with a system prompt
    messages = [{"role": "system",
                 "content": "You are a helpful, concise assistant."
                }
               ]
    
    # Appending whatever history the client provided:
    if provided_history:
        messages.extend(provided_history)
        
    # Appending the new prompt:
    messages.append({"role": "user",
                     "content": prompt
                    }
                   )
    
    # The LLM processes the entire chain of messages:
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=100
    )
    
    return response.choices[0].message.content.strip()

<font color="orange">To understand the limitation of a stateless agent, we simulate a simple user-model conversation through it:</font>

In [3]:
# --- Testing the Stateless Agent ---

print("--- Turn 1 ---")
prompt_1 = "Hi, my name is Alice and I am learning about API infrastructure."
response_1 = stateless_agent(prompt_1)
print(f"Agent: {response_1}")

print("\n--- Turn 2 (Without Client Context) ---")
# The agent fails here because it retained no memory of Turn 1
prompt_2 = "What is my name and what am I learning about?"
response_2 = stateless_agent(prompt_2)
print(f"Agent: {response_2}")

print("\n--- Turn 2 (With Client Context) ---")
# The frontend MUST inject the history into the payload for the agent to succeed
frontend_payload = [
    {"role": "user", "content": prompt_1},
    {"role": "assistant", "content": response_1}
]
response_3 = stateless_agent(prompt_2, provided_history=frontend_payload)
print(f"Agent: {response_3}")

--- Turn 1 ---
Agent: Hello Alice. I'd be happy to help you with your questions about API infrastructure. What specific topics would you like to discuss or learn more about? Gateway vs Edge services, API Security, API Gateway Architecture, Microservices Architecture, or something else? Let me know how I can assist you.

--- Turn 2 (Without Client Context) ---
Agent: Unfortunately, I don't have any information about you, including your name. I'm starting from a blank slate. However, we can chat and I can provide information on a wide range of topics.

--- Turn 2 (With Client Context) ---
Agent: Your name is Alice, and you are learning about API infrastructure.
